## Embedding Method - Feature Extraction.

Embedding training is a feature extraction method , we can say it is a deep learning method where we train the vector of each word per batch and get the optimum value through backpropagation. 

How are embeddings updated?
    
Standard flow :
- Input Sentence -> Embedding -> CNN -> Dense -> Softmax.
  The above is a standard flow of embedding + cnn deep learning model. 
    
- Creating an embedding matrix
  $$E \in \mathbb{R}^{V \times d}$$

- V = vocabulary size
- d = embedding dimension

Compute Loss: 
- if we use crossentropy = $$L = \text{crossentropy}(y, \hat{y})$$

Compute Gradients wrt loss and vector update: 
- $$\frac{\partial E[i]}{\partial L}$$
- gradient: $$g_i = \frac{\partial L}{\partial E[i]}$$
- If optimizer is SGD: $$E[i] := E[i] - \eta \cdot g_i$$
- If optimizer is ADAM : $$E[i] := E[i] - \eta \cdot \frac{m_i}{\sqrt{v_i} + \epsilon}$$

#### Model Training: 

Using this vectorizing method we will train Deep Learning Models:
- CNN - We will use a 1D Convolutional Model with kernel of dimension 3x1
- MLP - Using a standard MLP Architecture with 256 , 128 neurons in input and hidden layer.

## Embedding Method – Feature Extraction

Before training deep learning models, raw text must be converted into numeric form. We use the Tokenizer + Embedding method to achieve this.


$$
\text{Input} \;\rightarrow\; \text{Tokenizer} 
           \;\rightarrow\; \text{Embedding}
           \;\rightarrow\; \text{CNN / MLP}
           \;\rightarrow\; \text{Softmax}
$$


### Tokenizer

The Keras Tokenizer preprocesses and converts text into sequences of integers by:

- lowercasing text

- removing punctuation

- splitting into words (tokens)

- building a vocabulary of the top 𝑉 most frequent words

- mapping each word to an integer ID

- padding sequences to a fixed length

Each sentence becomes:
$$ sentence→[w1,w2,…,wT] $$
Embedding Layer

### An embedding matrix is initialized:

Where:

- $V$ = vocabulary size

- $𝑑$ = embedding dimension

Each integer token 
𝑖 retrieves vector $E[i]$.
These vectors are learned during training using backpropagation.

### Updating Embeddings

Given loss $𝐿$ (crossentropy):

$$
L = -\sum_{c} y_c \log(\hat{y}_c)
$$

SGD update:
$$
E[i] := E[i] - \eta \cdot g_i
$$


Adam update:
$$
E[i] := E[i] - \eta \cdot \frac{m_i}{\sqrt{v_i} + \epsilon}
$$

Only vectors corresponding to words in the batch are updated.

### Model Training

Using this vectorized representation, we train:

CNN Model

Embedding → Conv1D(kernel=3) → GlobalMaxPool → Dense → Softmax

MLP Model

Embedding → Dense(256) → Dense(128) → Dense(output)

Both models learn classification patterns using the embedding vectors as features.

In [1]:
import pandas as pd
import numpy as np
import random 
import re
from datasets import load_dataset
import time
from sklearn.metrics import accuracy_score, f1_score, classification_report

C:\Users\moham\anaconda3\envs\cuda-env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#loading dataset

dataset = load_dataset('ag_news')
train_df = dataset['train'].to_pandas()
test_df = dataset['test'].to_pandas()

X_train = train_df['text']
y_train = train_df['label']
X_test = test_df['text']
y_test = test_df['label']


In [3]:
X_train

0         Wall St. Bears Claw Back Into the Black (Reute...
1         Carlyle Looks Toward Commercial Aerospace (Reu...
2         Oil and Economy Cloud Stocks' Outlook (Reuters...
3         Iraq Halts Oil Exports from Main Southern Pipe...
4         Oil prices soar to all-time record, posing new...
                                ...                        
119995    Pakistan's Musharraf Says Won't Quit as Army C...
119996    Renteria signing a top-shelf deal Red Sox gene...
119997    Saban not going to Dolphins yet The Miami Dolp...
119998    Today's NFL games PITTSBURGH at NY GIANTS Time...
119999    Nets get Carter from Raptors INDIANAPOLIS -- A...
Name: text, Length: 120000, dtype: object

In [4]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

vocab_size = 65000            # enough for AG News
max_len = 200                 # sentence length
oov_token = "<OOV>"

tokenizer = Tokenizer(num_words=vocab_size, oov_token=oov_token)
tokenizer.fit_on_texts(X_train)

# convert text → integer sequences
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

# pad to fixed length
X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding='post')


In [5]:
X_train_seq

[[443,
  442,
  1682,
  14529,
  109,
  65,
  2,
  851,
  22,
  22,
  754,
  8197,
  443,
  6641,
  10232,
  2928,
  5,
  5811,
  25990,
  41,
  4050,
  798,
  333],
 [14992,
  1100,
  878,
  1304,
  4246,
  22,
  22,
  921,
  812,
  353,
  14992,
  100,
  103,
  23,
  4,
  4522,
  9,
  509,
  510,
  13268,
  7,
  14993,
  1521,
  2177,
  6,
  2,
  531,
  248,
  23,
  3937,
  2294,
  16,
  6561,
  8,
  213,
  369,
  5,
  2,
  129],
 [54,
  7,
  377,
  4577,
  25991,
  767,
  22,
  22,
  2427,
  463,
  91,
  1905,
  1284,
  67,
  2,
  377,
  7,
  2,
  767,
  9,
  285,
  41,
  192,
  3,
  5812,
  35,
  2,
  297,
  129,
  112,
  83,
  234,
  2,
  6205,
  5,
  2,
  1215,
  14994],
 [74,
  7405,
  54,
  1841,
  24,
  894,
  555,
  2900,
  22,
  22,
  861,
  36,
  4947,
  54,
  3601,
  7895,
  24,
  2,
  894,
  2900,
  6,
  555,
  74,
  29,
  1451,
  728,
  4,
  781,
  2582,
  92,
  515,
  3058,
  25,
  54,
  303,
  20,
  8,
  121],
 [54,
  91,
  4443,
  3,
  111,
  82,
  141,
  7473,
  18,


In [6]:
X_train_pad

array([[  443,   442,  1682, ...,     0,     0,     0],
       [14992,  1100,   878, ...,     0,     0,     0],
       [   54,     7,   377, ...,     0,     0,     0],
       ...,
       [ 7888,    60,   673, ...,     0,     0,     0],
       [ 4291,   782,   233, ...,     0,     0,     0],
       [ 2331,   226,  2515, ...,     0,     0,     0]])

In [7]:
from tensorflow.keras.utils import to_categorical

num_classes = 4  # AG News has 4 categories
y_train_oh = to_categorical(y_train, num_classes)
y_test_oh = to_categorical(y_test, num_classes)


In [8]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout

embedding_dim = 128
filters = 128
kernel_size = 3

cnn_model = Sequential([
    Embedding(vocab_size, embedding_dim, input_length=max_len, name="embedding"),

    Conv1D(filters=filters, kernel_size=kernel_size, activation='relu'),
    GlobalMaxPooling1D(),

    Dense(64, activation='relu'),
    Dropout(0.5),

    Dense(num_classes, activation='softmax')
])

cnn_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

cnn_model.summary()


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 200, 128)          8320000   
                                                                 
 conv1d (Conv1D)             (None, 198, 128)          49280     
                                                                 
 global_max_pooling1d (Globa  (None, 128)              0         
 lMaxPooling1D)                                                  
                                                                 
 dense (Dense)               (None, 64)                8256      
                                                                 
 dropout (Dropout)           (None, 64)                0         
                                                                 
 dense_1 (Dense)             (None, 4)                 260       
                                                        

In [9]:
%%time
cpu_start_time = time.process_time()
start_time = time.time()

history = cnn_model.fit(
    X_train_pad, y_train_oh,
    validation_split=0.1,
    epochs=5,
    batch_size=1000
)

cpu_end_time = time.process_time()
end_time = time.time()
cpu_time_train = cpu_end_time - cpu_start_time
cell_time_train =  end_time - start_time
print(cpu_time_train)
print(cell_time_train)

Epoch 1/5
108/108 [==============================] - 25s 223ms/step - loss: 0.7658 - accuracy: 0.7421 - val_loss: 0.2768 - val_accuracy: 0.9058
Epoch 2/5
108/108 [==============================] - 33s 304ms/step - loss: 0.2557 - accuracy: 0.9230 - val_loss: 0.2421 - val_accuracy: 0.9160
Epoch 3/5
108/108 [==============================] - 46s 421ms/step - loss: 0.1719 - accuracy: 0.9483 - val_loss: 0.2507 - val_accuracy: 0.9139
Epoch 4/5
108/108 [==============================] - 30s 283ms/step - loss: 0.1157 - accuracy: 0.9657 - val_loss: 0.2720 - val_accuracy: 0.9122
Epoch 5/5
108/108 [==============================] - 30s 277ms/step - loss: 0.0783 - accuracy: 0.9772 - val_loss: 0.3047 - val_accuracy: 0.9062
1685.796875
163.31832146644592
CPU times: total: 28min 5s
Wall time: 2min 43s


In [10]:
%%time
cpu_start_time = time.process_time()
start_time = time.time()

y_pred_probs = cnn_model.predict(X_test_pad)

cpu_end_time = time.process_time()
end_time = time.time()
cpu_time_inf = cpu_end_time - cpu_start_time
cell_time_inf =  end_time - start_time
print(cpu_time_inf)
print(cell_time_inf)

238/238 [==============================] - 1s 6ms/step
7.875
1.5171289443969727
CPU times: total: 7.88 s
Wall time: 1.52 s


In [11]:
y_pred_test = np.argmax(y_pred_probs, axis=1)

y_train_probs = cnn_model.predict(X_train_pad)
y_train_pred = np.argmax(y_train_probs,axis=1)

3750/3750 [==============================] - 22s 6ms/step


In [12]:
y_train_oh

array([[0., 0., 1., 0.],
       [0., 0., 1., 0.],
       [0., 0., 1., 0.],
       ...,
       [0., 1., 0., 0.],
       [0., 1., 0., 0.],
       [0., 1., 0., 0.]], dtype=float32)

In [13]:
y_train_pred

array([2, 2, 2, ..., 1, 1, 1], dtype=int64)

In [14]:
print("Training Accuracy")
train_accuracy = accuracy_score(y_train, y_train_pred)
train_f1 = f1_score(y_train, y_train_pred, average='weighted', zero_division=0)
train_report = classification_report(y_train, y_train_pred)
print('\n Training Accuracy',train_accuracy)
print('\n Training F1 Score',train_f1)
print('\n Training Classification Report',train_report)

Training Accuracy

 Training Accuracy 0.982125

 Training F1 Score 0.9821182280113712

 Training Classification Report               precision    recall  f1-score   support

           0       0.99      0.98      0.98     30000
           1       0.99      1.00      0.99     30000
           2       0.98      0.97      0.97     30000
           3       0.97      0.99      0.98     30000

    accuracy                           0.98    120000
   macro avg       0.98      0.98      0.98    120000
weighted avg       0.98      0.98      0.98    120000



In [15]:
print("Test Accuracy")
test_accuracy = accuracy_score(y_test, y_pred_test)
test_f1 = f1_score(y_test, y_pred_test, average='weighted', zero_division=0)
test_report = classification_report(y_test,y_pred_test)
print('\n Test Accuracy',test_accuracy)
print('\n Test F1 Score',test_f1)
print('\n Test Classification Report',test_report)

Test Accuracy

 Test Accuracy 0.9121052631578948

 Test F1 Score 0.9120221391223258

 Test Classification Report               precision    recall  f1-score   support

           0       0.92      0.91      0.91      1900
           1       0.96      0.97      0.97      1900
           2       0.89      0.87      0.88      1900
           3       0.88      0.90      0.89      1900

    accuracy                           0.91      7600
   macro avg       0.91      0.91      0.91      7600
weighted avg       0.91      0.91      0.91      7600



Per each sample ,the FLOPs carried can be calculated by the formula below. This is entirely dependent on the architecture.
$$FLOPs=2×(sequence length)×(kernel size)×(embedding dim)×(filters)$$

In [16]:
flops = 2 * 200 * 3 * 128 * 128
print("The FLOPs carried out per sample is: ", flops)

The FLOPs carried out per sample is:  19660800


In [17]:
results = {}
results[('Matrix Embedding','CNN Model')] = {
                                    'Train':
                                            {'accuracy': train_accuracy,
                                            'f1_score': train_f1,
                                            'report': train_report,
                                            'training_time': cpu_time_train},
                                    'Test':
                                            {'accuracy': test_accuracy,
                                            'f1_score': test_f1,
                                            'report': test_report,
                                            'inference_time':cpu_time_inf,
                                            "flops/sample": flops}}

### MLP Model:

In [18]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Flatten, Dense, Dropout
from tensorflow.keras import regularizers

embedding_dim = 128

mlp_model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len),
    Flatten(),
    Dense(256, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
    Dropout(0.3),
    Dense(128, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
    Dropout(0.3),
    Dense(4, activation='softmax')
])

mlp_model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

mlp_model.summary()


Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 200, 128)          8320000   
                                                                 
 flatten (Flatten)           (None, 25600)             0         
                                                                 
 dense_2 (Dense)             (None, 256)               6553856   
                                                                 
 dropout_1 (Dropout)         (None, 256)               0         
                                                                 
 dense_3 (Dense)             (None, 128)               32896     
                                                                 
 dropout_2 (Dropout)         (None, 128)               0         
                                                                 
 dense_4 (Dense)             (None, 4)                

In [19]:
%%time
cpu_start_time = time.process_time()
start_time = time.time() 

history_mlp = mlp_model.fit(
    X_train_pad,
    y_train_oh,
    validation_split=0.1,
    epochs=5,
    batch_size=128
)

cpu_end_time = time.process_time()
end_time = time.time()

cpu_train_time = cpu_end_time - cpu_start_time
cell_time = end_time - start_time
print(cpu_train_time)


Epoch 1/5
844/844 [==============================] - 56s 65ms/step - loss: 0.5440 - accuracy: 0.8514 - val_loss: 0.3692 - val_accuracy: 0.9097
Epoch 2/5
844/844 [==============================] - 53s 63ms/step - loss: 0.2738 - accuracy: 0.9470 - val_loss: 0.3782 - val_accuracy: 0.9051
Epoch 3/5
844/844 [==============================] - 53s 63ms/step - loss: 0.1957 - accuracy: 0.9737 - val_loss: 0.4332 - val_accuracy: 0.8968
Epoch 4/5
844/844 [==============================] - 53s 62ms/step - loss: 0.1405 - accuracy: 0.9861 - val_loss: 0.4521 - val_accuracy: 0.8913
Epoch 5/5
844/844 [==============================] - 53s 63ms/step - loss: 0.1231 - accuracy: 0.9904 - val_loss: 0.4944 - val_accuracy: 0.8906
2310.71875
CPU times: total: 38min 30s
Wall time: 4min 28s


In [20]:
%%time
cpu_start_time = time.process_time()
start_time = time.time()

y_pred_probs = mlp_model.predict(X_test_pad)

cpu_end_time = time.process_time()
end_time = time.time()
cpu_time_inf = cpu_end_time - cpu_start_time
cell_time_inf =  end_time - start_time
print(cpu_time_inf)
print(cell_time_inf)

238/238 [==============================] - 1s 6ms/step
7.25
1.4513561725616455
CPU times: total: 7.25 s
Wall time: 1.45 s


In [21]:
y_pred_test = np.argmax(y_pred_probs, axis=1)

y_train_probs = cnn_model.predict(X_train_pad)
y_train_pred = np.argmax(y_train_probs,axis=1)

3750/3750 [==============================] - 22s 6ms/step


In [22]:
print("Training Accuracy")
train_accuracy = accuracy_score(y_train, y_train_pred)
train_f1 = f1_score(y_train, y_train_pred, average='weighted', zero_division=0)
train_report = classification_report(y_train, y_train_pred)
print('\n Training Accuracy',train_accuracy)
print('\n Training F1 Score',train_f1)
print('\n Training Classification Report',train_report)

Training Accuracy

 Training Accuracy 0.982125

 Training F1 Score 0.9821182280113712

 Training Classification Report               precision    recall  f1-score   support

           0       0.99      0.98      0.98     30000
           1       0.99      1.00      0.99     30000
           2       0.98      0.97      0.97     30000
           3       0.97      0.99      0.98     30000

    accuracy                           0.98    120000
   macro avg       0.98      0.98      0.98    120000
weighted avg       0.98      0.98      0.98    120000



In [23]:
print("Test Accuracy")
test_accuracy = accuracy_score(y_test, y_pred_test)
test_f1 = f1_score(y_test, y_pred_test, average='weighted', zero_division=0)
test_report = classification_report(y_test,y_pred_test)
print('\n Test Accuracy',test_accuracy)
print('\n Test F1 Score',test_f1)
print('\n Test Classification Report',test_report)

Test Accuracy

 Test Accuracy 0.8994736842105263

 Test F1 Score 0.8996236037292534

 Test Classification Report               precision    recall  f1-score   support

           0       0.93      0.88      0.90      1900
           1       0.96      0.96      0.96      1900
           2       0.88      0.85      0.86      1900
           3       0.84      0.90      0.87      1900

    accuracy                           0.90      7600
   macro avg       0.90      0.90      0.90      7600
weighted avg       0.90      0.90      0.90      7600



Per each sample ,the FLOPs carried can be calculated by the formula below. This is entirely dependent on the architecture.
$$FLOPs=2×(sequence length)×(kernel size)×(embedding dim)×(filters)$$

In [24]:
def analytical_flops(model):
    flops = 0
    for layer in model.layers:
        if isinstance(layer, tf.keras.layers.Dense):
            in_features = layer.input_shape[-1]
            out_features = layer.output_shape[-1]
            flops += 2 * in_features * out_features
    return flops

flops = analytical_flops(mlp_model)
print(f"Analytical FLOPs: {flops} FLOPs")


Analytical FLOPs: 13173760 FLOPs


In [25]:
results[('Matrix Embedding','MLP Model')] = {
                                    'Train':
                                            {'accuracy': train_accuracy,
                                            'f1_score': train_f1,
                                            'report': train_report,
                                            'training_time': cpu_time_train},
                                    'Test':
                                            {'accuracy': test_accuracy,
                                            'f1_score': test_f1,
                                            'report': test_report,
                                            'inference_time':cpu_time_inf,
                                            "flops/sample": flops}}

In [26]:
# create DataFrame
import pandas as pd

# flatten nested dict into a list of rows
rows = []

for (vectorizer, model), metrics in results.items():
    row = {
        "Vectorizer": vectorizer,
        "Model": model,
        "Train_Accuracy": metrics["Train"]["accuracy"],
        "Train_F1": metrics["Train"]["f1_score"],
        "Test_Accuracy": metrics["Test"]["accuracy"],
        "Test_F1": metrics["Test"]["f1_score"],
        "Inference_Time": metrics["Test"].get("inference_time", None),
        "FLOPs_per_Sample": metrics["Test"].get("flops/sample", None),
        "Training_Time": metrics['Train'].get('training_time',None)
    }
    rows.append(row)

# create DataFrame
df_results = pd.DataFrame(rows)

# optional: set a clean display order
df_results = df_results[
    ["Vectorizer", "Model", "Train_Accuracy", "Train_F1",
     "Test_Accuracy", "Test_F1", "Inference_Time", "FLOPs_per_Sample","Training_Time"]
]

print(df_results)

         Vectorizer      Model  Train_Accuracy  Train_F1  Test_Accuracy  \
0  Matrix Embedding  CNN Model        0.982125  0.982118       0.912105   
1  Matrix Embedding  MLP Model        0.982125  0.982118       0.899474   

    Test_F1  Inference_Time  FLOPs_per_Sample  Training_Time  
0  0.912022           7.875          19660800    1685.796875  
1  0.899624           7.250          13173760    1685.796875  


In [27]:
results = pd.read_csv('datasets/results.csv')

df = pd.concat([results, df_results], axis=0,ignore_index=True)

In [28]:
df = df.drop('Unnamed: 0',axis=1)
df

,Vectorizer,Model,Train_Accuracy,Train_F1,Test_Accuracy,Test_F1,Inference_Time,FLOPs_per_Sample,Training_Time
0,BOW,Logistic Model,0.929450,0.929328,0.902237,0.902075,0.136527,10000.0,4.875000
1,BOW,Random Forest Model,0.789375,0.788256,0.781579,0.779980,0.210410,1000.0,7.531250
2,BOW,XG Boost Model,0.930625,0.930504,0.904605,0.904392,0.184776,4000.0,63.421875
3,BOW,MLP Model,0.949333,0.949257,0.913026,0.912820,1.236788,2626560.0,53.453125
4,BOW,Scratch MLP Model,0.920692,0.920560,0.902895,0.902761,3.453125,2626560.0,1562.906250
5,Tf_IDF,Logistic Model,0.919283,0.919100,0.905658,0.905387,0.302151,10000.0,10.062500
6,Tf_IDF,Random Forest Model,0.778833,0.779300,0.768684,0.768673,0.341379,1000.0,17.015625
7,Tf_IDF,XG Boost Model,0.936292,0.936206,0.903026,0.902790,0.376454,4000.0,2439.328125
8,Tf_IDF,MLP Model,0.930417,0.930338,0.905263,0.905156,1.405875,2626560.0,49.234375
9,Tf_IDF,Scratch MLP Model,0.896383,0.896034,0.888026,0.887549,3.312500,2626560.0,1473.250000


In [29]:
df.to_csv('datasets/results.csv')
df_results.to_csv('datasets/matrix_emb.csv')